第三版，导入MCC zener_diode 的表。

如果是query chip出现NA，会动态的调整fit模型时用到的参数。（默认权重）

如果是candidate chip出现NA，会用中位数代替。

待解决：zener diodes没有categorical，需要改推荐模型

In [44]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.metrics.pairwise import cosine_similarity

df=pd.read_csv(
    'MCC_product_lists/MCC_DataExport_zener-diodes(MCC-zener-diodes).csv',
    header=0,
    skiprows=[0,2]
)

df.columns = [' '.join(col.split()) for col in df.columns]

df_selected=df.iloc[:,[1,3,4]+list(range(6,16))]
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 2144 entries, 0 to 2143
Data columns (total 16 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   Manufacture          2144 non-null   str    
 1   Product              2144 non-null   str    
 2   Status               2144 non-null   str    
 3   Compliance           2144 non-null   str    
 4   Number of Functions  2144 non-null   str    
 5   Configuration        0 non-null      float64
 6   Package Type         2144 non-null   str    
 7   PD(W)                2144 non-null   float64
 8   VZ[Nom](V)           2144 non-null   float64
 9   IZT(mA)              2144 non-null   float64
 10  IR (mA) [max] @VR    2144 non-null   float64
 11  VR(V)                2144 non-null   float64
 12  ZZT(Ω) @IZT          2105 non-null   float64
 13  ZZK(Ω) @IZT          2071 non-null   float64
 14  IZK(mA)              2071 non-null   float64
 15  Tj [max] (°C)        2144 non-null   int64  
dtyp

In [59]:
df[df['PD(W)']==0.5].head(10)

,Manufacture,Product,Status,Compliance,Number of Functions,Configuration,Package Type,PD(W),VZ[Nom](V),IZT(mA),IR (mA) [max] @VR,VR(V),ZZT(Ω) @IZT,ZZK(Ω) @IZT,IZK(mA),Tj [max] (°C)
67,MCC (Micro Commercial Components),BZT52A24,Active,R H,Single,NaN,SOD-123,0.5,24.0,5.0,0.045,16.8,65.0,235.0,1.00,150
190,MCC (Micro Commercial Components),MMSZ5221B,Active,R H,Single,NaN,SOD-123,0.5,2.4,20.0,0.100,1.0,30.0,1200.0,0.25,150
191,MCC (Micro Commercial Components),MMSZ5222B,Active,R H,Single,NaN,SOD-123,0.5,2.5,20.0,0.100,1.0,30.0,1250.0,0.25,150
192,MCC (Micro Commercial Components),MMSZ5223B,Active,R H,Single,NaN,SOD-123,0.5,2.7,20.0,0.075,1.0,30.0,1300.0,0.25,150
193,MCC (Micro Commercial Components),MMSZ5225B,Active,R H,Single,NaN,SOD-123,0.5,3.0,20.0,0.050,1.0,29.0,1600.0,0.25,150
194,MCC (Micro Commercial Components),MMSZ5226B,Active,R H,Single,NaN,SOD-123,0.5,3.3,20.0,0.025,1.0,28.0,1600.0,0.25,150
195,MCC (Micro Commercial Components),MMSZ5227B,Active,R H,Single,NaN,SOD-123,0.5,3.6,20.0,0.015,1.0,24.0,1700.0,0.25,150
196,MCC (Micro Commercial Components),MMSZ5228B,Active,R H,Single,NaN,SOD-123,0.5,3.9,20.0,0.010,1.0,23.0,1900.0,0.25,150
197,MCC (Micro Commercial Components),MMSZ5229B,Active,R H,Single,NaN,SOD-123,0.5,4.3,20.0,0.005,1.0,22.0,2000.0,0.25,150
198,MCC (Micro Commercial Components),MMSZ5230B,Active,R H,Single,NaN,SOD-123,0.5,4.7,20.0,0.005,2.0,19.0,1900.0,0.25,150


In [45]:
df_selected.head()

,Product,Compliance,Number of Functions,Package Type,PD(W),VZ[Nom](V),IZT(mA),IR (mA) [max] @VR,VR(V),ZZT(Ω) @IZT,ZZK(Ω) @IZT,IZK(mA),Tj [max] (°C)
0,3SMAJ5929BQ,A R H,Single,SMA,3.0,15.0,25.0,0.001,11.4,9.0,600.0,0.25,150
1,3SMAJ5933BQ,A R H,Single,SMA,3.0,22.0,17.0,0.001,16.7,17.5,650.0,0.25,150
2,3SMAJ5935BQ,A R H,Single,SMA,3.0,27.0,13.9,0.001,20.6,23.0,700.0,0.25,150
3,3SMAJ5936BQ,A R H,Single,SMA,3.0,30.0,12.5,0.001,22.8,28.0,750.0,0.25,150
4,3SMAJ5937BQ,A R H,Single,SMA,3.0,33.0,11.4,0.001,25.1,33.0,800.0,0.25,150


In [46]:
def apply_hard_constraints(candidate_chips, query_chip): 
    # 从 query_chip 中提取标量值
    package = query_chip['Package Type'].iloc[0]

    filtered = candidate_chips[
        (candidate_chips['Package Type'] == package)
    ].copy()
    
    
    return filtered

In [47]:
all_numeric_features = [
    'PD(W)',
    'VZ[Nom](V)',
    'IZT(mA)',
    'IR (mA) [max] @VR',
    'VR(V)',
    'ZZT(Ω) @IZT',
    'ZZK(Ω) @IZT',
    'IZK(mA)',
    'Tj [max] (°C)'
]

categorical_features = ['Number of Functions']

# 非特征的列（不参与相似度计算）
exclude_columns = ['Product', 'Number of Functions', 'Package Type']

In [48]:
def get_available_features(query_chip, all_features):
    """
    检查 query_chip 中哪些特征列不是 NA，返回可用特征列表
    """
    available = []
    for col in all_features:
        if col in query_chip.columns:
            # 检查该列的第一个值是否为 NA
            if pd.notna(query_chip[col].iloc[0]):
                available.append(col)
            else:
                print(f"⚠️ 跳过特征 '{col}'（query_chip 中为 NA）")
        else:
            print(f"⚠️ 跳过特征 '{col}'（不在 query_chip 中）")
    return available

In [49]:
def create_dynamic_preprocessor(df_inventory, query_chip, all_features):
    """
    根据 query_chip 的缺失情况，动态创建预处理流水线
    """
    # 获取可用的特征
    available_features = get_available_features(query_chip, all_features)
    
    if not available_features:
        raise ValueError("query_chip 中没有任何可用特征！")
    
    print(f"✅ 使用的数值特征 ({len(available_features)}个):", available_features)
    
    # 创建预处理器（只对可用特征进行预处理）
    preprocessor = ColumnTransformer([
        ('num', Pipeline([
            ('imputer', SimpleImputer(strategy='median')),  # 候选芯片缺失值用中位数填充
            ('scaler', StandardScaler())
        ]), available_features),
        ('cat', OneHotEncoder(handle_unknown='ignore'), categorical_features)
    ])
    
    # 在完整数据集上 fit
    preprocessor.fit(df_inventory)
    
    return preprocessor, available_features

In [50]:
def create_query_chip(columns, values_dict):
    """
    创建 query_chip，缺失值用 np.nan 表示
    """
    # 从 df_selected 获取列名
    columns_list = columns.tolist() if hasattr(columns, 'tolist') else columns
    
    # 按照列顺序构建值列表
    values = []
    for col in columns_list:
        if col in values_dict:
            values.append(values_dict[col])
        else:
            values.append(np.nan)  # 没有指定的列用 NA
    
    return pd.DataFrame([values], columns=columns_list)

In [51]:
def weighted_similarity(query_vec, candidate_matrix, feature_weights):
    """
    计算加权余弦相似度
    feature_weights: 列表，顺序与编码后的特征列顺序一致
    """
    # 先对候选矩阵的每一列乘以对应的权重
    weighted_candidates = candidate_matrix * feature_weights
    weighted_query = query_vec * feature_weights
    
    # 计算余弦相似度
    similarities = cosine_similarity(weighted_query, weighted_candidates)
    return similarities.flatten()

In [52]:
def recommend_alternatives_dynamic(df_inventory, query_chip, all_features, top_k=10, weights=None):
    """
    动态推荐函数 - 根据 query_chip 的可用特征自动调整
    """
    # 1. 硬约束过滤
    candidates = apply_hard_constraints(df_inventory, query_chip)
    if candidates.empty:
        return "没有配置/封装完全匹配的替代品"
    
    # 2. 动态创建预处理器
    preprocessor, available_features = create_dynamic_preprocessor(
        df_inventory, query_chip, all_features
    )
    
    # 3. 编码
    query_vec = preprocessor.transform(query_chip)
    # print(f"编码后的总特征数: {query_vec.shape[1]}")
    candidate_vecs = preprocessor.transform(candidates)
    
    # 4. 计算相似度
    if weights is None:
        weights = np.ones(candidate_vecs.shape[1])
    else:
        # 如果传入的权重长度与特征数不匹配，调整
        if len(weights) != candidate_vecs.shape[1]:
            print(f"⚠️ 权重长度 ({len(weights)}) 与特征数 ({candidate_vecs.shape[1]}) 不匹配，使用默认权重")
            weights = np.ones(candidate_vecs.shape[1])
    
    scores = weighted_similarity(query_vec, candidate_vecs, weights)
    
    # 5. 排序并返回
    candidates = candidates.copy()
    candidates['similarity_score'] = scores
    results = candidates.sort_values('similarity_score', ascending=False).head(top_k)

    # 添加车规级标识列
    # 假设 df_inventory 中有 'Compliance' 列
    results['Is_Automotive'] = results['Compliance'].apply(
        lambda x: '车规级' if pd.notna(x) and 'A' in str(x) else '非车规级'
    )
    
    # 返回结果（包含动态特征）
    return_columns = ['Product', 'similarity_score','Is_Automotive'] + available_features + categorical_features
    return results[return_columns]

# def recommend_alternatives_dynamic(df_inventory, query_chip, all_features, top_k=10, weights=None):
#     """
#     动态推荐函数 - 根据 query_chip 的可用特征自动调整
#     """
#     # 1. 硬约束过滤
#     candidates = apply_hard_constraints(df_inventory, query_chip)
#     if candidates.empty:
#         return "没有配置/封装完全匹配的替代品"
    
#     # 2. 动态创建预处理器
#     preprocessor, available_features = create_dynamic_preprocessor(
#         df_inventory, query_chip, all_features
#     )
    
#     # 3. 编码
#     query_vec = preprocessor.transform(query_chip)
#     candidate_vecs = preprocessor.transform(candidates)
    
#     # 4. 自动生成权重
#     if weights is None:
#         weights = np.ones(candidate_vecs.shape[1])
#     else:
#         # 如果传入的权重长度与特征数不匹配，自动补全
#         if len(weights) != candidate_vecs.shape[1]:
#             n_numeric = len(available_features)  # 数值特征数量（如 9）
#             n_categorical = candidate_vecs.shape[1] - n_numeric  # 分类特征数量
            
#             print(f"📊 总特征数: {candidate_vecs.shape[1]}, 数值特征: {n_numeric}, 分类特征: {n_categorical}")
            
#             if len(weights) == n_numeric:
#                 # 只传了数值特征的权重，自动补充分类特征权重
#                 categorical_weight = 0.3
#                 weights = np.concatenate([
#                     np.array(weights),
#                     np.full(n_categorical, categorical_weight)
#                 ])
#                 print(f"✅ 自动补全权重: 数值权重 {len(weights) - n_categorical} 个, 分类权重 {n_categorical} 个 (全部设为 {categorical_weight})")
#             else:
#                 # 长度完全不匹配，使用默认权重
#                 print(f"⚠️ 权重长度 ({len(weights)}) 与总特征数 ({candidate_vecs.shape[1]}) 不匹配，使用默认权重")
#                 weights = np.ones(candidate_vecs.shape[1])
    
#     scores = weighted_similarity(query_vec, candidate_vecs, weights)
    
#     # 5. 排序并返回
#     candidates = candidates.copy()
#     candidates['similarity_score'] = scores
#     results = candidates.sort_values('similarity_score', ascending=False).head(top_k)

#     # 添加车规级标识列
#     results['Is_Automotive'] = results['Compliance'].apply(
#         lambda x: '车规级' if pd.notna(x) and 'A' in str(x) else '非车规级'
#     )
    
#     # 返回结果（包含动态特征）
#     return_columns = ['Product', 'similarity_score', 'Is_Automotive'] + available_features + categorical_features
#     return results[return_columns]

In [ ]:
# 构建 query_chip（所有参数都有值）
columns = df_selected.columns.tolist()

# query_values = { #BZX84B5V1HE3（正确答案是BZX84C5V1HE3-TP
#     'Product': 'SZBZX84C5V1ET1G',
#     'Number of Functions': 'Single',
#     'Package Type': 'SOT-23',
#     'PD(W)':0.25,
#     'VZ[Nom](V)':5.1,
#     'IZT(mA)':5,
#     'IR (mA) [max] @VR':0.002,
#     'VR(V)':2,
#     'ZZT(Ω) @IZT':60,
#     'ZZK(Ω) @IZT':None,
#     'IZK(mA)':None,
#     'Tj [max] (°C)':150
# }

query_values = { #BZX84C3V3HE3（正确答案是BZX84C3V3HE3-TP
    'Product': 'BZX84-C3V3,235',
    'Number of Functions': 'Single',
    'Package Type': 'SOT-23',
    'PD(W)':0.25,
    'VZ[Nom](V)':3.3,
    'IZT(mA)':5,
    'IR (mA) [max] @VR':0.005,
    'VR(V)':1,
    'ZZT(Ω) @IZT':95,
    'ZZK(Ω) @IZT':600,
    'IZK(mA)':1,
    'Tj [max] (°C)':150
}

query_chip = create_query_chip(columns, query_values)

# 权重
# 'Reverse Standoff Voltage VRWM(V)': 2.0,   # 硬约束参数，非常重要
# 'Peak Pulse Current IPP(A)': 1.8,          # 硬约束参数，重要
# 'Max. Clamping Voltage VC (V)': 2.0,       # 硬约束参数，非常重要
# 'Junction Capacitance CJ(pF)': 1.8,        # 硬约束参数，重要
# 'Peak Pluse Power Dissipation PPPK (W)': 1.0,  # 参考值
# 'Maximum Reverse Leakage IR (uA)': 0.8,    # 不那么关键
# 'Breakdown Voltage Min VBR(V)': 1.2,       # 与VRWM相关，次重要
# 'Breakdown Voltage Max VBR(V)': 1.2,       # 与VRWM相关，次重要
# 'Junction Temperature Tj [max] (°C)': 0.5  # 大多数都是150，区分度低
custom_weights = np.array([2.0, 1.8, 2.0, 1.5, 1.0, 0.8, 1.2, 1.2, 0.5])

# 执行推荐（所有9个数值特征都会使用）
result = recommend_alternatives_dynamic(df_selected, query_chip, all_numeric_features, top_k=5)
print(result)

⚠️ 跳过特征 'ZZT(Ω) @IZT'（query_chip 中为 NA）
⚠️ 跳过特征 'ZZK(Ω) @IZT'（query_chip 中为 NA）
✅ 使用的数值特征 (7个): ['PD(W)', 'VZ[Nom](V)', 'IZT(mA)', 'IR (mA) [max] @VR', 'VR(V)', 'IZK(mA)', 'Tj [max] (°C)']
           Product  similarity_score Is_Automotive  PD(W)  VZ[Nom](V)  \
1497  BZX84C3V6HE3          0.999146           车规级   0.35         3.6   
225      BZX84C3V6          0.999146          非车规级   0.35         3.6   
2138  BZX84B3V6HE3          0.999146           车规级   0.35         3.6   
1450     BZX84B3V6          0.999146          非车规级   0.35         3.6   
1496  BZX84C3V3HE3          0.999133           车规级   0.35         3.3   

      IZT(mA)  IR (mA) [max] @VR  VR(V)  IZK(mA)  Tj [max] (°C)  \
1497      5.0              0.005    1.0      1.0            150   
225       5.0              0.005    1.0      1.0            150   
2138      5.0              0.005    1.0      1.0            150   
1450      5.0              0.005    1.0      1.0            150   
1496      5.0              0.005    1